# Quantum Rings Challenge - Exploratory Data Analysis

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load Data
data_path = Path('data/hackathon_public.json')
with open(data_path, 'r') as f:
    data = json.load(f)

circuits = pd.DataFrame(data['circuits'])
results = pd.DataFrame(data['results'])

## 1. Data Structure Check

In [ ]:
print("Circuits shapeData:", circuits.shape)
print("Results shape:", results.shape)
display(circuits.head(2))
display(results.head(2))

## 2. Derive True Labels
We need to find:
1. `true_threshold_min`: The lowest threshold where fidelity >= 0.99.
2. `forward_runtime`: The runtime at the selected threshold.

In [ ]:
def get_true_labels(row):
    # 1. Find Minimum Threshold
    threshold_sweep = row['threshold_sweep']
    true_min = None
    
    # Sort just in case, though usually sorted
    sorted_sweep = sorted(threshold_sweep, key=lambda x: x['threshold'])
    
    count_passed = 0
    for run in sorted_sweep:
        # Use sdk_get_fidelity or p_return_zero? challenge docs say sdk_get_fidelity closely tracks p_return_zero.
        # The target is 0.99
        fid = run.get('sdk_get_fidelity')
        if fid is not None and fid >= 0.99:
            true_min = run['threshold']
            break
            
    # If none met, use the max threshold attempted (or mark as failed)
    if true_min is None:
        if row['status'] == 'no_threshold_met':
             # Use the max threshold as a proxy or handle separately
             true_min = sorted_sweep[-1]['threshold']
        else:
             # If status is OK but we didn't find it in sweep? Weird.
             pass

    # 2. Get Forward Runtime
    # Creating the target variable. Docs say: "predict the expected wall-clock runtime for a fixed-shot forward simulation"
    # We want 'run_wall_s' from the 'forward' object.
    forward_time = None
    fwd = row.get('forward')
    if fwd and 'run_wall_s' in fwd:
        forward_time = fwd['run_wall_s']
        
    return pd.Series([true_min, forward_time], index=['true_threshold', 'true_runtime'])

y = results.apply(get_true_labels, axis=1)
results = pd.concat([results, y], axis=1)

## 3. Visualize Distribution

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.countplot(x='true_threshold', data=results)
plt.title('Distribution of True Minimum Threshold')

plt.subplot(1, 2, 2)
sns.histplot(results['true_runtime'], log_scale=True)
plt.title('Distribution of Forward Runtime (Log Scale)')

plt.show()